In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [11]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [12]:
# ========================
# LOAD DATA (AMAN DI COLAB)
# ========================
data = pd.read_csv("/content/gdrive/MyDrive/Test/WineQT.csv")

print(data.head())

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  Id  
0      9.4        5   0  
1      9.8        5   1  
2      9

In [13]:
# ========================
# PREPROCESSING
# ========================
# Cek kolom (biar aman kalau beda nama)
print("\nKolom:", data.columns)


Kolom: Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality', 'Id'],
      dtype='object')


In [14]:
# Drop kolom jika ada
columns_to_drop = []
if "quality" in data.columns:
    target = "quality"
else:
    raise Exception("Kolom 'quality' tidak ditemukan!")

if "Id" in data.columns:
    columns_to_drop.append("Id")

X = data.drop(columns_to_drop + [target], axis=1)
y = data[target]

In [15]:
# ========================
# HOLDOUT
# ========================
print("\n=== HOLDOUT ===")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    error = 1 - accuracy

    print(f"K = {k}")
    print(f"Akurasi = {accuracy * 100:.2f}%")
    print(f"Error = {error * 100:.2f}%")
    print("----------------------")


=== HOLDOUT ===
K = 1
Akurasi = 60.70%
Error = 39.30%
----------------------
K = 3
Akurasi = 50.22%
Error = 49.78%
----------------------
K = 5
Akurasi = 51.53%
Error = 48.47%
----------------------


In [16]:
# ========================
# K-FOLD
# ========================
print("\n=== K-FOLD (5 Fold) ===")

for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(knn, X, y, cv=5)

    accuracy = scores.mean()
    error = 1 - accuracy

    print(f"K = {k}")
    print(f"Akurasi rata-rata = {accuracy * 100:.2f}%")
    print(f"Error rata-rata = {error * 100:.2f}%")
    print("----------------------")


=== K-FOLD (5 Fold) ===
K = 1
Akurasi rata-rata = 45.40%
Error rata-rata = 54.60%
----------------------
K = 3
Akurasi rata-rata = 43.74%
Error rata-rata = 56.26%
----------------------
K = 5
Akurasi rata-rata = 45.15%
Error rata-rata = 54.85%
----------------------


In [17]:
# ========================
# RANDOM SUBSAMPLING
# ========================
print("\n=== RANDOM SUBSAMPLING ===")

errors = {1: [], 3: [], 5: []}

for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    for k in [1, 3, 5]:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train)

        y_pred = knn.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        error = 1 - accuracy

        errors[k].append(error)

for k in [1, 3, 5]:
    print(f"K = {k}, Error rata-rata = {np.mean(errors[k]) * 100:.2f}%")

# ========================
# LEAVE ONE OUT
# ========================
print("\n=== LEAVE ONE OUT ===")

loo = LeaveOneOut()

for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(knn, X, y, cv=loo)

    accuracy = scores.mean()
    error = 1 - accuracy

    print(f"K = {k}")
    print(f"Akurasi = {accuracy * 100:.2f}%")
    print(f"Error = {error * 100:.2f}%")
    print("----------------------")


=== RANDOM SUBSAMPLING ===
K = 1, Error rata-rata = 44.02%
K = 3, Error rata-rata = 52.45%
K = 5, Error rata-rata = 49.65%

=== LEAVE ONE OUT ===
K = 1
Akurasi = 58.71%
Error = 41.29%
----------------------
K = 3
Akurasi = 48.73%
Error = 51.27%
----------------------
K = 5
Akurasi = 49.34%
Error = 50.66%
----------------------


In [18]:
# ========================
# BOOTSTRAP
# ========================
print("\n=== BOOTSTRAP ===")

errors = {1: [], 3: [], 5: []}

for i in range(10):
    sample = data.sample(frac=1, replace=True)
    X_sample = sample.drop(columns_to_drop + [target], axis=1)
    y_sample = sample[target]

    X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2)

    for k in [1, 3, 5]:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train)

        y_pred = knn.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        error = 1 - accuracy

        errors[k].append(error)

for k in [1, 3, 5]:
    print(f"K = {k}, Error rata-rata = {np.mean(errors[k]) * 100:.2f}%")


=== BOOTSTRAP ===
K = 1, Error rata-rata = 23.80%
K = 3, Error rata-rata = 36.81%
K = 5, Error rata-rata = 41.92%
